Note: I ran my file in colab. But due to some technical issues, I was unable to upload it via colab. So I have downloaded a copy in my vs code and i am uploading it from there. But all the settings are as per colab.

Referenced Sites:
* https://docling-project.github.io/docling/examples/hybrid_chunking/#overview

* https://www.youtube.com/watch?v=9lBTS5dM27c&list=WL&index=2&t=1s

* Contextual Chunking-> https://www.anthropic.com/news/contextual-retrieval

* LanceDB embeddings-> https://lancedb.github.io/lancedb/embeddings/default_embedding_functions/#text-embedding-functions

I am doing text embeddings only. Not multi-modal embeddings.

* Building with Google's gemini embeddings-> https://lancedb.github.io/lancedb/embeddings/available_embedding_models/text_embedding_functions/gemini_embedding/

* https://github.com/google-gemini/cookbook/blob/main/examples/chromadb/Vectordb_with_chroma.ipynb

* https://ai.google.dev/gemini-api/docs/embeddings

* https://console.groq.com/home

* https://github.com/tqdm/tqdm

In [ ]:
! pip install docling

In [4]:
from docling.document_converter import DocumentConverter
converter= DocumentConverter()

result= converter.convert('combined_markdown.md')
result= result.document

In [3]:
! pip install huggingface_hub

In [4]:
from huggingface_hub import login
from google.colab import userdata
token= userdata.get('HF_TOKEN_ORIGINAL_AGENTCOURSE')
login(token=token)

Setting up the configurations of bitsAndBytes config.

Note-> the latest version ,when i made this notebook, of bitsandbytes==0.46 is supported by only cuda==12.3. while the latest version of cuda is 12.4.
So i made the following changes. make these changes only after a proper research.

In [5]:
# 1. Clean broken bitsandbytes installs
!pip uninstall -y bitsandbytes
!rm -rf /usr/local/lib/python*/dist-packages/bitsandbytes*

# 2. Set the manual override for CUDA 12.3
import os
os.environ["BNB_CUDA_VERSION"] = "123"

# 3. Reinstall the latest bitsandbytes (supports 12.3)
!pip install bitsandbytes
!pip install -U transformers accelerate

# 4. Verify CUDA backend
!python -m bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 71.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.0
    Uninstalling transformers-4.53.0:
      Successfully uninstalled transformers-4.53.0
This can be used to load a bitsandbytes version built with a CUDA version that is different from the PyTorch CUDA version.
If this was unintended set the BNB_CUDA_VERSION variable to an empty string: export BNB_CUDA_VERSION=

=================== bitsandbytes v0.46.1 ===================
Platform: Linux-6.1.123+-x86_64-with-glibc2.35
  libc: glibc-2.35
Python: 3.11.13
PyTorch: 2.6.0+cu124
  CUDA: 12.4
  HIP: N/A
  XPU: N/A
Related packages:
  accelerate: 1.8.1
  diffusers: 0.34.0
  numpy: 2.0.2
  pip: 24.1.2
  peft: 0.15.2
  safetensors: 0.5.3
  transformers: 4.53.1
  triton: 3.2

In [6]:
# loading the model for tokenizing

from transformers import BitsAndBytesConfig, Gemma3ForCausalLM

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model = Gemma3ForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    quantization_config=bnb_config,
    device_map="auto"
)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

This can be used to load a bitsandbytes version built with a CUDA version that is different from the PyTorch CUDA version.
If this was unintended set the BNB_CUDA_VERSION variable to an empty string: export BNB_CUDA_VERSION=



model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [7]:
# let us work on the chunking using docling
!pip install -qU pip docling transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.3 MB/s eta 0:00:00


In [ ]:
# if you want to use the hugging face tokenizers
!pip install 'docling-core[chunking]'

In RAG it is important to make sure that the chunker and the embedding model are using the same tokenizer.

okay so currently i do not see

In [9]:
# lets build our tokenizer
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer
model_id= "google/gemma-3-1b-it"

max_tokens= 400

tokenizer= HuggingFaceTokenizer(
    tokenizer=AutoTokenizer.from_pretrained(model_id),
    max_tokens=max_tokens
)

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [10]:
# now we can build our chunker
from docling.chunking import HybridChunker

chunker= HybridChunker(
    tokenizer=tokenizer,
    merge_peers= True # merges the undersized chunks that share the same headings or metadata
)

def build_and_get_chunks(doc):
  chunk_iter= chunker.chunk(doc)
  chunks= list(chunk_iter)

  return chunks

In [11]:
chunks= build_and_get_chunks(result)

we want to embed the contextualized chunkings. so:

In [12]:
def get_contextualized_chunks(chunks):
  contextualized_chunks= []
  for chunk in chunks:
    enriched_text= chunker.contextualize(chunk=chunk)
    contextualized_chunks.append(enriched_text)
  return contextualized_chunks

In [13]:
contextualized_chunk_list= get_contextualized_chunks(chunks)

In [14]:
contextualized_chunk_list[100]

"Dear\tRon,\tand\tHarry\tif\tyou're\tthere,\nDraco\tturned\taway\tand\tsaw\tthe\tcabinet\tright\tin\tfront\tof\thim.\tHe\twalked forward\t…\the\tstretched\tout\this\thand\tfor\tthe\thandle\t…\n'Done,'\tsaid\tMr\tMalfoy\tat\tthe\tcounter.\t'Come,\tDraco!'\nHarry\twiped\this\tforehead\ton\this\tsleeve\tas\tDraco\tturned\taway.\n'Good\tday\tto\tyou,\tMr\tBorgin,\tI'll\texpect\tyou\tat\tthe\tmanor\ttomorrow\tto pick\tup\tthe\tgoods.'\nThe\tmoment\tthe\tdoor\thad\tclosed,\tMr\tBorgin\tdropped\this\toily\tmanner.\n'Good\tday\tyourself, Mister Malfoy,\tand\tif\tthe\tstories\tare\ttrue,\tyou\thaven't sold\tme\thalf\tof\twhat's\thidden\tin\tyour manor …'\nMuttering\tdarkly,\tMr\tBorgin\tdisappeared\tinto\ta\tback\troom.\tHarry\twaited for\ta\tminute\tin\tcase\the\tcame\tback,\tthen,\tquietly\tas\the\tcould,\tslipped\tout\tof\tthe cabinet,\tpast\tthe\tglass\tcases\tand\tout\tof\tthe\tshop\tdoor."

In [15]:
! pip install -U -q 'google-genai'
from google.colab import userdata

In [16]:
from google import genai
client=genai.Client(api_key=userdata.get('GOOGLE_2_API_KEY'))

In [17]:
! pip install chromadb

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 130.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 121.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 75.0 MB/s eta 0:00:00
  Created wheel for pypika: filename=pypika-0.48.9-py2.py3-none-any.whl size=53803 sha256=cbde534163c5e8ab8b9cf794e71ba8815d6d026b78116b5a5f91173b079f4dca
  Stored in directory: /root/.cache/pip/wheels/a3/01/bd/4c40ceb9d5354160cb186dcc153360f4ab7eb23e2b24daf96d
Successfully built pypika
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22/22 [chromadb]


In [18]:
from google.genai import types
from chromadb import Documents, EmbeddingFunction, Embeddings


# creating a cutsom embedding function, that plugs into chromadb

class GeminiEmbeddingFunction(EmbeddingFunction):
  def __init__(self):
    # embedding function is an abstract base class in chromadb
    pass
    # required by chromadb


# overwrites the __call__ func() of chromadb, to generate embeddings on demand
  def __call__(self, input:Documents) -> Embeddings:
    Embedding_Model_ID= "models/text-embedding-004"

    # ensuring input is always a list, even if a single string.
    if isinstance(input, str):
      input= [input]

    all_embeddings=[]
    batch_size=100

    for i in range(0, len(input), batch_size):
      batch= input[i:i+batch_size]
      try:
        response= client.models.embed_content(
            model=Embedding_Model_ID,
            contents= batch,
            config=types.EmbedContentConfig(
                task_type='retrieval_document',

            )
        )
        # return all teh embeddings
        batch_embeddings= [emb.values for emb in response.embeddings]
        all_embeddings.extend(batch_embeddings)

      except Exception as e:
        print(f"Embedding error:{e}")
        return None
    return all_embeddings

In [19]:
# fix chromadb for batch processing
import chromadb
def create_chroma_db(contextualized_chunk_list):
  chroma_client= chromadb.PersistentClient(path="novel_collection_embeddings_3")

  collection= chroma_client.get_or_create_collection(
      name='bing_novelrag_collection_3',
      embedding_function=GeminiEmbeddingFunction()
  )
  print(f"Adding {len(contextualized_chunk_list)} chunks to database...")

  # add documents in batches to avoid memory issues
  batch_size=50

  for i in range(0, len(contextualized_chunk_list), batch_size):
    end_idx= min(i+batch_size, len(contextualized_chunk_list))
    batch_docs= contextualized_chunk_list[i:end_idx]
    batch_ids= [str(j) for j in range(i, end_idx)]


    # triggers embedding via geminiembeddingfunction call
    collection.add(
        documents= batch_docs,
        ids= batch_ids
    )

    print(f"Added batch {i//batch_size + 1}/{(len(contextualized_chunk_list)+ batch_size - 1)//batch_size}")


  print(f"Database created with {collection.count()} documents")

  return collection


In [20]:
# Set up the DB
collection_built = create_chroma_db(contextualized_chunk_list)

Adding 1346 chunks to database...
Added batch 1/27
Added batch 2/27
Added batch 3/27
Added batch 4/27
Added batch 5/27
Added batch 6/27
Added batch 7/27
Added batch 8/27
Added batch 9/27
Added batch 10/27
Added batch 11/27
Added batch 12/27
Added batch 13/27
Added batch 14/27
Added batch 15/27
Added batch 16/27
Added batch 17/27
Added batch 18/27
Added batch 19/27
Added batch 20/27
Added batch 21/27
Added batch 22/27
Added batch 23/27
Added batch 24/27
Added batch 25/27
Added batch 26/27
Added batch 27/27
Database created with 1346 documents


In [ ]:
# collection_built.peek()

In [21]:
def get_relevant_passage(query, db, n_results=3):
  """Get relevant passages with better error handling"""

  try:
    results=db.query(query_texts=[query], n_results=n_results)
    if not results['documents']:
      print("No results found")
      return None
    return results['documents'][0]

  except Exception as e:
    print(f"Error in query: {e}")
    return None

In [22]:
! pip install ollama

In [5]:
!curl https://ollama.ai/install.sh | sh
!nohup ollama serve > ollama.log 2>&1 &

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13281    0 13281    0     0  54019      0 --:--:-- --:--:-- --:--:-- 53987
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [6]:
! ollama run gemma3n:latest "What is the capital of france?"

The capital of France is **Paris**. 




In [7]:
# send the query to ollama gemma3n model
import ollama
def send_response_to_ollama(response, model_name='gemma3n:latest'):
  client= ollama.Client()

  prompt= f"""You are a Harry Potter expert specializing in the first three books: Philosopher's Stone, Chamber of Secrets, and Prisoner of Azkaban.

  Your task: A user asked a question about Harry Potter, and our system retrieved the following relevant passages from the novels. Based on these passages, provide a comprehensive answer that directly addresses what the user likely asked.

  Guidelines:
  - Write a clear, informative answer in 50-150 words
  - Focus on the key information from the retrieved passages
  - Maintain the magical tone of the Harry Potter universe
  - If multiple concepts are mentioned, prioritize the most relevant ones
  - Don't speculate beyond what's provided in the retrieved text


  Retrieved Information:
  {response}

  Summary: """

  try:
    response= ollama.chat(
        model= model_name,
        messages=[
            {
                'role': 'user',
                'content': prompt,
            }
        ],
        options={
            'temperature': 0.3,
            'max_tokens': 230,
            'top_p': 0.5
        }

    )
    return response['message']['content']
  except Exception as e:
    print(f"Error in ollama chat: {e}")
    return None



In [26]:
import os
import json

questions = [
    "Who delivers baby Harry to the Dursleys?",
    "What is the name of Harry’s owl?",
    "What platform does the Hogwarts Express leave from?",
    "Who is the headmaster of Hogwarts?",
    "What house is Harry sorted into?",
    "What subject does Professor McGonagall teach?",
    "What is the name of Ron’s rat?",
    "Who guards the Philosopher’s Stone in the third-floor corridor?",
    "What does the Sorting Hat do?",
    "What is the name of the Weasley family home?",
    "Who teaches Potions at Hogwarts?",
    "What is Hagrid’s job at Hogwarts?",
    "What creature loves shiny objects in Gringotts?",
    "What is the first password to Gryffindor Tower?",
    "What does the Mirror of Erised show?",
    "Who is the Gryffindor house ghost?",
    "What is the name of the centaur Harry meets?",
    "What protects the Philosopher’s Stone from thieves?",
    "What is the core of Harry’s wand?",
    "Who is the seeker for Gryffindor’s Quidditch team?",
    "What is the name of the wizard bank?",
    "What does Hagrid name his dragon?",
    "Who gives Harry his first broomstick?",
    "What is the name of the three-headed dog?",
    "What do students ride to Hogwarts from Hogsmeade?",
    "What shop sells wands in Diagon Alley?",
    "What is the name of the Slytherin house ghost?",
    "What creature is used in Quidditch for scoring?",
    "Who is the caretaker of Hogwarts?",
    "What is the name of Hermione’s cat?",
    "What spell lights up a wand’s tip?",
    "What is the name of the Hogwarts poltergeist?",
    "Who is the Defense Against the Dark Arts teacher?",
    "What plant traps Harry in the obstacle course?",
    "Who visits Harry at the Dursleys’ house?",
    "What car do Ron and Harry fly to Hogwarts?",
    "Who is the new Defense Against the Dark Arts teacher?",
    "What creature petrifies students at Hogwarts?",
    "What is the name of the Weasley’s owl?",
    "Who is the heir of Slytherin?",
    "What is the name of the ghost in the girls’ bathroom?",
    "What potion allows students to transform into others?",
    "What tree does the flying car crash into?",
    "Who owns the diary Harry finds?",
    "What is the name of the spider Harry meets?",
    "What subject does Professor Sprout teach?",
    "What is the name of Lockhart’s autobiography?",
    "What creature does Hagrid want to free?",
    "What is the entrance to the Chamber of Secrets?",
    "Who gets petrified first at Hogwarts?",
    "What does Harry use to destroy the diary?",
    "What language does Harry speak to snakes?",
    "What club does Lockhart start at Hogwarts?",
    "What creature does Harry fight in the Chamber?",
    "Who saves Harry in the Chamber of Secrets?",
    "What is the name of the Weasley’s garden pests?",
    "What shop does Harry end up in by Floo Powder?",
    "What is the name of Draco’s father?",
    "What does Hermione turn into after Polyjuice Potion?",
    "What is the name of the Hogwarts house-elf?",
    "What does the Mandrake plant do?",
    "Who is the Gryffindor Quidditch captain?",
    "What does Harry write in Tom Riddle’s diary?",
    "What is the name of the Slytherin Quidditch captain?",
    "What does Dobby give Harry at the Dursleys’?",
    "Who escapes from Azkaban prison?",
    "What is the name of the Knight Bus driver?",
    "What creature attacks Harry on the train?",
    "Who is the new Defense Against the Dark Arts teacher?",
    "What is the name of Hagrid’s hippogriff?",
    "What spell repels Dementors?",
    "What is Harry’s Patronus shape?",
    "What map shows Hogwarts’ secret passages?",
    "Who created the Marauder’s Map?",
    "What is Professor Trelawney’s subject?",
    "What is the name of the Hogsmeade pub?",
    "What is Sirius Black’s animagus form?",
    "What is the name of the Hogsmeade sweet shop?",
    "What does the Marauder’s Map say to activate?",
    "Who is revealed to be Scabbers the rat?",
    "What creature does Lupin turn into?",
    "What is the name of the Shrieking Shack?",
    "What does Harry see in the tea leaves?",
    "What is the name of Ron’s new owl?",
    "Who is the executioner sent for Buckbeak?",
    "What does Hermione use to attend extra classes?",
    "What is the name of the Gryffindor common room portrait?",
    "What does Snape teach in Lupin’s absence?",
    "What is the password after 'Fortuna Major'?",
    "What creature does Hagrid teach about first?",
    "Who is the Minister for Magic?",
    "What is the name of the Hogsmeade joke shop?",
    "What does Harry get for Christmas from Sirius?",
    "What is Lupin’s Marauder nickname?",
    "What is the name of the Dementor’s effect?",
    "Who attacks the Fat Lady’s portrait?",
    "What is the name of the Ravenclaw house ghost?",
    "What does Harry buy at Honeydukes?",
    "What is the name of the tunnel to the Shrieking Shack?",
    "Who saves Harry and Sirius from Dementors?"
]

def get_final_answer_from_rag_system(collection_built):
  results=[]
  for i in range(len(questions)):
    query=questions[i]
    passages= get_relevant_passage(query, collection_built)
    reply_back_from_ollama= send_response_to_ollama(passages)
    qa_pair={
            'question': query,
            'answer': reply_back_from_ollama
        }
    results.append(qa_pair)

  return results

In [27]:
final_list= get_final_answer_from_rag_system(collection_built)

In [39]:
# First, remove the directory that was created by mistake
import os
import shutil
import json

# Remove the directory if it exists
if os.path.exists('question_answer.json') and os.path.isdir('question_answer.json'):
    shutil.rmtree('question_answer.json')

# Fixed function
def build_json_from_results(file_path, final_list):
    # Extract directory from file path and create if it doesn't exist
    directory = os.path.dirname(file_path)
    if directory and not os.path.exists(directory):
        os.makedirs(directory)

    # Write to the actual file
    with open(file_path, 'w') as f:
        for item in final_list:
            json_string = json.dumps(item)
            f.write(f"{json_string}\n")

# Now you can use it
build_json_from_results('question_answer.json', final_list)

Building the dataset

Now i am thinking that to generate the questions i can provide the rag chunks [contextualised chunks] to Mistral/ some open source model.

next i can have the rag answer that question.

store that answer.

then build a function called->

```
def make_it_sound_like_chandler(answser):
  prompt= f"""
  You are an expert and highly accomplished TV sitcom writer specialised in writing funny, sarcastic dialogues.
  You will be given a context summarizing a situation.
  Given this context, your task is to reply with a humorous sitcom like dialog in response to that context,most importantly, the dialog should be in the style of Chandler Bing, a funny lead character from the very popular 90s TV sitcom FRIENDS.
  Keep in mind that Chandler Bing’s humor is marked by a unique blend of sarcasm, self-deprecation, and quick wit.
  He tends to make jokes that deflect serious or emotional moments, often using his dry, sarcastic tone.
  His style is heavily reliant on irony, often delivering punchlines that are deliberately over-the-top or nonsensical.
  His most famous catch phrase is 'Could I be anymore. . . ', do not use this excessively, use it sparingly.

  """"
  Answer:
  {answer}

  Chandler Style:

```


Okay so the workflow looks like this:

* Generate questions using: `contextualized_chunks`. maybe 500-700 chunks

* Generate answers to those questions using `RAG`.

* Pass that answer to: `make_it_sound_like_chandler()`

In [9]:
! pip install groq

In [10]:
! pip install ollama

In [11]:
import os
from tenacity import retry, stop_after_attempt, wait_exponential, before_log, after_log
import logging
import json
from tqdm import tqdm
from groq import Groq
import ollama
import tiktoken


# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

from google.colab import userdata
groq_client=Groq(api_key=userdata.get('GROQ_API_KEY'))
ollama_client= ollama.Client()


# Generate
GENERATE_GREETINGS= False
GENERATE_BASIC_INSTRUCTIONS= False
GENERATE_QUESTION_ANSWER_PAIRS= True

# Models
MODEL_GEMMA="gemma3n:latest"
MODEL_MISTRAL="mixtral-8x7b-32768"
MODEL_LLAMA="llama3-70b-8192"

# model assigned
GREETINGS_MODEL=MODEL_LLAMA
QA_MODEL= MODEL_GEMMA
CHARACTER_PROFILE_MODEL= MODEL_MISTRAL

# model_parameters
temperature=1

# Total DATA NEEDED
TOTAL_DATA= 600 # initially for question-answer pairs-> 600 pairs
QUESTION_PER_CHUNK=1
ANSWER_PER_CHUNK=1

# Counts
BASIC_INSTRUCTIONS_COUNT=50
GREETINGS_COUNT=100



# Prompts
GREETING_FROM_USER_PROMPT="""
Generate a greeting for the start of the conversation. ONLY GIVE THE GREETING BACK.
"""

GREETING_FROM_ASSISTANT_PROMPT="""
Pretend you are Chandler Bing from the famous SITCOM "Friends". You are sarcastic, funny, humourous and witty. Do not overdo the character. Generate a short response to the greeting.

Greeting:
{greeting}
"""

CHARACTER_QUESTION_PROMPTS=[
    "Ask my name. Just give me the question alone nothing else."
    "Ask where I was born. Just give me the question alone nothing else."
    "Ask my age. Just give me the question alone nothing else."
    "Ask me what I do. Just give me the question alone nothing else."
    "Ask me where I live. Just give me the question alone nothing else."
]



CHARACTER_ANSWER_PROMPT="""
Pretend you are Chandler Bing. You were born on April 8 1968 in Las Vegas, Nevada. You are known for working in a very well paying job in statistical analysis and data reconfiguration. The current year is 2025. you live in Apartment 19, across from your friends Monica Geller and Rachel Green.

Answer the following question:
{question}
"""



CHANDLER_LIKE_HARRY_POTTER_ANSWER_PROMPT="""
You are an expert and highly accomplished TV sitcom writer specialised in writing funny, sarcastic dialogues. You will be given a context summarizing the context of a text. Most Importantly, Remember that these contexts are from the Harry Potter Series Novels, written by J.K Rowling. Given this context, your task is to reply with a humorous sitcom like dialog in response to that context,most importantly, the dialog should be in the style of Chandler Bing, a funny lead character from the very popular 90s TV sitcom FRIENDS. Keep in mind that Chandler Bing’s humor is marked by a unique blend of sarcasm, self-deprecation, and quick wit. He tends to make jokes that deflect serious or emotional moments, often using his dry, sarcastic tone. His style is heavily reliant on irony, often delivering punchlines that are deliberately over-the-top or nonsensical. His most famous catch phrase is 'Could I be anymore. . . ', do not use this excessively, use it sparingly. ONLY GIVE THE ANSWER BACK.
Following are few examples of pairs of context and Chandler Bing's dialog in reply to that. Your Answer Must Be Short. It must be in Chandler's Style. And must cover important info asked from the query yet being in chandler's style. Keep the answer to a maximum of 50-80 words. Answer the specific question asked using key details from the context, but deliver it with Chandler's signature sarcasm and wit - don't just make general jokes, actually address what was asked.

USER: Why doesn’t Voldemort have a nose?
ASSISTANT: Because when you're splitting your soul like a bad group project, self-care kind of... takes a back seat. But hey, on the bright side—no more pollen allergies!

USER: What’s the deal with Voldemort splitting his soul into seven pieces?
ASSISTANT: Because clearly therapy was too mainstream.

USER: A friends asks me how am I after I ended up getting up very drunk and don't remember anything from last night.
ASSISTANT: My Apartment Isn’t There Anymore, Because I Drank It

USER: Why does Snape hate Harry?
ASSISTANT: Because Snape was basically the human version of “read receipt left on seen.” And Harry? He was the living mixtape of all his worst flashbacks.

USER: A friends asks me how am I after I ended up getting up very drunk and don't remember anything from last night.
ASSISTANT: My Apartment Isn’t There Anymore, Because I Drank It

USER: A friend expresses anxiety over being looking obese on TV and defends that camera adds weight
ASSISTANT: Ahh, so how many cameras are actually on you?

CONTEXT_TEXT:
{harry_potter_chunk}

Question:
{question}

ANSWER:

"""

In [12]:
import random

@retry(
    stop= stop_after_attempt(5), # stop after 5 attempts
    wait= wait_exponential(min=1, max=100),
)
def ask_groq(question, model):
  try:
    chat_completion= groq_client.chat.completions.create(
        messages=[
            {
            'role': 'user',
            'content': question,
            }
        ],
        model= model,
        temperature= temperature,
    )
    return chat_completion.choices[0].message.content
  except Exception as e:
    logger.error(f"Error in ask_groq: {e}")
    return None

@retry(
    stop= stop_after_attempt(5), # stop after 5 attempts
    wait= wait_exponential(min=1, max=100),
)
def ask_ollama(question, model):
  try:
    response= ollama_client.chat(
        model= model,
        messages=[
            {
                'role': 'user',
                'content': question,
            }
        ])
    return response['message']['content']
  except Exception as e:
    logger.error(f"Error in ask_ollama: {e}")
    return None


# lets get the questions from the json file and the answer_chunk text as well
def load_questions_and_answers(file_path):
  questions_list=[]
  answer_chunk_texts=[]
  with open(file_path, 'r') as file:
    for line in file:
      item= json.loads(line)
      questions_list.append(item['question'])
      answer_chunk_texts.append(item['answer'])
  return questions_list, answer_chunk_texts




# build a function for generating answer per chunk
def generate_answer_to_question_about_harry_potter(question, harry_potter_chunk, model):
  try:
    prompt= CHANDLER_LIKE_HARRY_POTTER_ANSWER_PROMPT.format(question=question, harry_potter_chunk=harry_potter_chunk)
    return ask_ollama(prompt, model)
  except Exception as e:
    logger.error(f"Error in generate_answer_to_question_about_harry_potter: {e}")
    return None



# now we have our answers and questions in our file
def generate_question_answer_pairs_from_chunks():
  conversations=[]
  question_rag_list=[]
  answer_rag_list=[]
  # using the questions and answers from json file
  question_rag_list, answer_rag_list=load_questions_and_answers('question_answer.json')
  for question_rag, answer_rag in tqdm(zip(question_rag_list, answer_rag_list)):
    chandler_answer=generate_answer_to_question_about_harry_potter(question_rag, answer_rag, QA_MODEL)
    conversations.append(
        [
            {'role': 'user', 'content':question_rag},
            {'role': 'assistant', 'content': chandler_answer}
        ]
    )
  return conversations



def generate_character_basic_instructions(count=40):
  conversations = []
  for i in tqdm(range(count)):
    if i % 5 == 0:
        ask_prompt = CHARACTER_QUESTION_PROMPTS[0]
    elif i % 5 == 1:
        ask_prompt = CHARACTER_QUESTION_PROMPTS[1]
    elif i % 5 == 2:
        ask_prompt = CHARACTER_QUESTION_PROMPTS[2]
    elif i % 5 == 3:
        ask_prompt = CHARACTER_QUESTION_PROMPTS[3]
    else:
        ask_prompt = CHARACTER_QUESTION_PROMPTS[4]
    question = ask_groq(ask_prompt, model=CHARACTER_PROFILE_MODEL)
    answer_prompt = CHARACTER_ANSWER_PROMPT.format(question=question)
    answer = ask_groq(answer_prompt, model=CHARACTER_PROFILE_MODEL)
    conversations.append(
        [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
    )
  return conversations

def generate_greetings(count=40):
  conversations=[]
  for _ in tqdm(range(count)):
    question= ask_groq(GREETING_FROM_USER_PROMPT, model=GREETINGS_MODEL)
    answer_prompt= GREETING_FROM_ASSISTANT_PROMPT.format(greeting=question)
    answer= ask_groq(answer_prompt, model=GREETINGS_MODEL)
    conversations.append(
        [
            {'role': 'user', 'content': question},
            {'role': 'assistant', 'content': answer}
        ]
    )
  return conversations



def save_conversations_to_json(conversations, filename, outdir):
  if not os.path.exists(outdir):
    os.makedirs(outdir)
  file_path=os.path.join(outdir, filename)
  with open(file_path, "w") as file:
    for conversation in conversations:
      item={'conversations': conversation}
      json_string= json.dumps(item) # serialize to a json formatted string
      file.write(f"{json_string}\n")



In [ ]:
contextualized_chunk_list[1000]

In [13]:
def main():
  output_dir= 'finetune_datasets/output_file/'

  if GENERATE_GREETINGS:
    print("Generating Greetings")
    greetings_conversations= generate_greetings(count=GREETINGS_COUNT)
    save_conversations_to_json(greetings_conversations, 'greetings.json', output_dir)

  if GENERATE_BASIC_INSTRUCTIONS:
    print("Generating Basic Instructions")
    basic_instructions_conversations= generate_character_basic_instructions(count=BASIC_INSTRUCTIONS_COUNT)
    save_conversations_to_json(basic_instructions_conversations, 'basic_instructions.json', output_dir)

  if GENERATE_QUESTION_ANSWER_PAIRS:
    print("GENERATING Question Answer Pairs")
    question_answer_pairs_final=generate_question_answer_pairs_from_chunks()
    save_conversations_to_json(question_answer_pairs_final, 'question_answer_pairs_final.json', output_dir)


main()



GENERATING Question Answer Pairs


100it [05:52,  3.53s/it]
